In [204]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [205]:
data=pd.read_csv('heart.csv')
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [206]:
data.tail()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0


In [207]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [208]:
data.shape

(303, 14)

In [209]:
data.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [210]:
data.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')

In [211]:
X=data.drop(columns='target')
Y=data['target']

In [212]:
numerical_columns=['age','trestbps','chol','thalach','oldpeak']
cat_columns=[col for col in data.columns if col not in numerical_columns]

In [213]:
for col in cat_columns:
    print(f'\n{col}')
    print(data[col].unique())


sex
[1 0]

cp
[3 2 1 0]

fbs
[1 0]

restecg
[0 1 2]

exang
[0 1]

slope
[0 2 1]

ca
[0 2 1 3 4]

thal
[1 2 3 0]

target
[1 0]


In [214]:
cat_cols_encoder=['cp','restecg','thal']
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
encoded_data=encoder.fit_transform(X[cat_cols_encoder])
encoded_df=pd.DataFrame(encoded_data,columns=encoder.get_feature_names_out(cat_cols_encoder),
                       index=X.index)
X=pd.concat([X.drop(columns=cat_cols_encoder),encoded_df],axis=1)

In [215]:
X.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,slope,ca,...,cp_1,cp_2,cp_3,restecg_0,restecg_1,restecg_2,thal_0,thal_1,thal_2,thal_3
0,63,1,145,233,1,150,0,2.3,0,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,37,1,130,250,0,187,0,3.5,0,0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,41,0,130,204,0,172,0,1.4,2,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,56,1,120,236,0,178,0,0.8,2,0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,57,0,120,354,0,163,1,0.6,2,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [216]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y, test_size=0.2, random_state=42)

In [217]:
def tune_clf_hyperparameters(clf, param_grid,X_train,Y_train, scoring='recall', n_splits=5):
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=30)
    clf_grid=GridSearchCV(clf,param_grid,cv=cv,scoring=scoring,n_jobs=-1)
    clf_grid.fit(X_train,Y_train)
    best_hyperparameters=clf_grid.best_params_
    return clf_grid.best_estimator_,best_hyperparameters

### Model 1: Logistic Regression

In [219]:
prep = ColumnTransformer(
    [("num", StandardScaler(), numerical_columns)],
    remainder="passthrough",
)

model_1 = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])

In [220]:
param_grid_1={ "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced", {0: 1, 1: 2}, {0: 1, 1: 3}]
}

In [221]:
best_model_1,best_model_1_hyperparameters=tune_clf_hyperparameters(model_1,param_grid_1,X_train,Y_train)
print('Model_1 Optimal Hyperparameters: \n',best_model_1_hyperparameters)

Model_1 Optimal Hyperparameters: 
 {'clf__C': 0.001, 'clf__class_weight': None}


In [222]:
print(classification_report(Y_train,best_model_1.predict(X_train)))

              precision    recall  f1-score   support

           0       1.00      0.26      0.41       109
           1       0.62      1.00      0.77       133

    accuracy                           0.67       242
   macro avg       0.81      0.63      0.59       242
weighted avg       0.79      0.67      0.61       242



In [223]:
print(classification_report(Y_test,best_model_1.predict(X_test)))

              precision    recall  f1-score   support

           0       0.92      0.41      0.57        29
           1       0.65      0.97      0.78        32

    accuracy                           0.70        61
   macro avg       0.78      0.69      0.67        61
weighted avg       0.78      0.70      0.68        61



In [224]:
def model_evaluation(model,X_test,Y_test,model_name):
    Y_pred=model.predict(X_test)
    report=classification_report(Y_test,Y_pred,output_dict=True)
    metrics={
        "precision_0" :report['0'] ["precision"],
         "precision_1" :report['1'] ["precision"],
         "recall_0" :report['0'] ["recall"],
         "recall_1" :report['1'] ["recall"],
        "f1_0": report["0"]["f1-score"],
        "f1_1": report["1"]["f1-score"],
        "accuracy": accuracy_score(Y_test, Y_pred)
    }
    data=pd.DataFrame(metrics,index=[model_name]).round(2)
    return data

In [225]:
df_evaluation_1=model_evaluation(best_model_1,X_test,Y_test,'Logistic Regression')
df_evaluation_1

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,accuracy
Logistic Regression,0.92,0.65,0.41,0.97,0.57,0.78,0.7


### Model 2: Decision Tree

In [227]:
param_grid_2={
    'criterion':['gini','entropy'],
    'max_depth':[2,3,4,5],
    'min_samples_split':[2,3,4],
    'min_samples_leaf':[2,4,6,7,8]
}

In [228]:
model_2=DecisionTreeClassifier(random_state=42)

In [229]:
best_model_2,best_model_2_hyperparameters=tune_clf_hyperparameters(model_2,param_grid_2,X_train,Y_train)
print('Model_2 Optimal Hyperparameters: \n',best_model_2_hyperparameters)

Model_2 Optimal Hyperparameters: 
 {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 6, 'min_samples_split': 2}


In [230]:
print(classification_report(Y_test,best_model_2.predict(X_test)))

              precision    recall  f1-score   support

           0       0.77      0.83      0.80        29
           1       0.83      0.78      0.81        32

    accuracy                           0.80        61
   macro avg       0.80      0.80      0.80        61
weighted avg       0.81      0.80      0.80        61



In [231]:
df_evaluation_2=model_evaluation(best_model_2,X_test,Y_test,'Decision Tree')
df_evaluation_2

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,accuracy
Decision Tree,0.77,0.83,0.83,0.78,0.8,0.81,0.8


### Model 3: Random Forest

In [233]:
model_3=RandomForestClassifier(random_state=42)

In [234]:
param_grid_3={
    'n_estimators':[10,20,30,40],
    'criterion':['gini','entropy'],
    'max_depth':[2,3,4],
    'min_samples_split':[2,3,4,5],
    'min_samples_leaf':[1,2,3],
    'bootstrap':[True,False]
}

In [235]:
best_model_3,best_model_3_hyperparameters=tune_clf_hyperparameters(model_3,param_grid_3,X_train,Y_train)
print('Model_3 Optimal Hyperparameters: \n',best_model_3_hyperparameters)

Model_3 Optimal Hyperparameters: 
 {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 30}


In [236]:
print(classification_report(Y_test,best_model_3.predict(X_test)))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88        29
           1       0.88      0.91      0.89        32

    accuracy                           0.89        61
   macro avg       0.89      0.88      0.88        61
weighted avg       0.89      0.89      0.89        61



In [237]:
df_evaluation_3=model_evaluation(best_model_3,X_test,Y_test,'Random Forest')
df_evaluation_3

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,accuracy
Random Forest,0.89,0.88,0.86,0.91,0.88,0.89,0.89


### Model 4: KNN

In [239]:
prep1 = ColumnTransformer(
    [("num", StandardScaler(), numerical_columns)],
    remainder="passthrough",
)

model_4 = Pipeline([
    ("prep", prep1),
    ("clf", KNeighborsClassifier()),
])

In [240]:
param_grid_4={
    'clf__n_neighbors': list(range(1, 12)),
    'clf__weights': ['uniform', 'distance'],
    'clf__p': [1, 2]
}

In [241]:
best_model_4,best_model_4_hyperparameters=tune_clf_hyperparameters(model_4,param_grid_4,X_train,Y_train)
print('Model_4 Optimal Hyperparameters: \n',best_model_4_hyperparameters)

Model_4 Optimal Hyperparameters: 
 {'clf__n_neighbors': 11, 'clf__p': 1, 'clf__weights': 'uniform'}


In [242]:
print(classification_report(Y_test,best_model_4.predict(X_test)))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88        29
           1       0.88      0.91      0.89        32

    accuracy                           0.89        61
   macro avg       0.89      0.88      0.88        61
weighted avg       0.89      0.89      0.89        61



In [243]:
df_evaluation_4=model_evaluation(best_model_4,X_test,Y_test,'KNN')
df_evaluation_4

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,accuracy
KNN,0.89,0.88,0.86,0.91,0.88,0.89,0.89


In [244]:
all_evaluations=[df_evaluation_1, df_evaluation_2, df_evaluation_3, df_evaluation_4]
result=pd.concat(all_evaluations)
result=result.sort_values(by='recall_1',ascending=False)
result

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,accuracy
Logistic Regression,0.92,0.65,0.41,0.97,0.57,0.78,0.70
Random Forest,0.89,0.88,0.86,0.91,0.88,0.89,0.89
KNN,0.89,0.88,0.86,0.91,0.88,0.89,0.89
Decision Tree,0.77,0.83,0.83,0.78,0.80,0.81,0.80


#### Model 1 (Logistic Regression) is good at identifying people who may have heart disease.

#### With a recall of 0.97 for class 1, it correctly identifies 97% of patients who actually have heart disease.

#### It also maintains good precision, helping reduce false alarms and unnecessary alerts.

### https://colab.research.google.com/drive/13du2cnay5bj3tnMTiG6JrExlPMx0YNvC?usp=sharing